# Test `predict` on 1 claim
Cost-controlled smoke test: loads a single row from `claims.csv`, builds the prompt + image blocks, and calls the model once.

Requires credits on the Anthropic account for the API key in `.env`.

In [1]:
import sys, os
# run from repo root and make the code/ package importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'code'))

from loaders import load_claims
from images import build_image_blocks
from prompt import build_system_prompt, build_user_message
from model_client import predict

In [5]:
# 1 item only - keep cost minimal
claim = load_claims('/Users/xieqiqi/Learning/assignment/hackerrank-orchestrate-june26/dataset/sample_claims.csv')[0]
print(claim.user_id, '|', claim.claim_object, '|', len(claim.images), 'images')

image_blocks, missing = build_image_blocks(claim.images)
print('image blocks:', len(image_blocks), '| missing:', missing)

user_001 | car | 1 images
image blocks: 2 | missing: []


In [6]:
system = build_system_prompt()
user_content = build_user_message(
    claim,
    history_summary='Low-risk user with prior accepted vehicle claims.',
    evidence_text='The claimed object and relevant part should be clearly visible.',
    image_blocks=image_blocks,
)

In [7]:
result, usage = predict(system, user_content)
print('--- RESULT ---')
for k, v in result.model_dump().items():
    print(f'{k}: {v}')
print('--- USAGE ---')
print(usage)

--- RESULT ---
user_id: 
image_paths: 
user_claim: New dent on the rear bumper area of the car after being parked outside overnight
claim_object: car
evidence_standard_met: yes
evidence_standard_met_reason: The image clearly shows the rear of the car, including the trunk, taillights, and bumper area, which are the relevant parts to the claim.
risk_flags: none
issue_type: dent/structural body damage
object_part: rear bumper and trunk/tailgate area
claim_status: supported
claim_status_justification: In img_1, the rear of the silver car shows significant body damage: a large dent and crumpling across the trunk lid, the rear bumper cover is missing or torn away exposing the bumper reinforcement bar, and the surrounding panels are deformed. This clearly matches the claimed dent in the rear bumper area.
supporting_image_ids: img_1
valid_image: yes
severity: high
--- USAGE ---
{'input': 582, 'output': 320, 'cache_read': 1305, 'cache_write': 0}
